In [1]:
import duckdb
import pandas as pd

In [2]:
con = duckdb.connect()

In [3]:
df = con.sql("""
--I want to understand which customers drive most of the revenue.
WITH clean AS(    
    SELECT CustomerID, Quantity * UnitPrice AS revenue
    FROM 'online_retail.csv'
    WHERE
    CustomerID IS NOT NULL
    AND Quantity > 0
    AND UnitPrice > 0
), 
total AS(
    SELECT CustomerID, SUM(revenue) AS total_sales, AVG(revenue) AS average_sales, COUNT(*) AS total_orders
    FROM clean
    GROUP BY CustomerID
),
--The thresholds for segments are determined as median values. 
segment AS(
    SELECT CustomerID, total_sales, average_sales, total_orders, MEDIAN(average_sales) OVER() AS median_sales, MEDIAN(total_orders) OVER() AS median_orders, 
        CASE WHEN average_sales > MEDIAN(average_sales) OVER() THEN 'high_sales' ELSE 'low_sales' END AS sales_category,
        CASE WHEN total_orders >MEDIAN(total_orders) OVER()THEN 'high_orders' ELSE 'low_orders' END AS orders_category, 
        --SUM(total_sales) OVER(ORDER BY total_sales DESC) * 100.0 / SUM(total_sales) OVER() AS cumulative_percentage
    FROM total
)
SELECT SUM(CASE WHEN sales_category = 'high_sales' AND orders_category = 'high_orders' THEN total_sales ELSE 0 END) * 100.0 / SUM(total_sales) AS high_sales_high_orders,
    SUM(CASE WHEN sales_category = 'low_sales' AND orders_category = 'high_orders' THEN total_sales ELSE 0 END) * 100.0 / SUM(total_sales) AS high_sales_low_orders,
    SUM(CASE WHEN sales_category = 'high_sales' AND orders_category = 'low_orders' THEN total_sales ELSE 0 END) * 100.0 / SUM(total_sales) AS low_sales_high_orders,
    SUM(CASE WHEN sales_category = 'low_sales' AND orders_category = 'low_orders' THEN total_sales ELSE 0 END) * 100.0 / SUM(total_sales) AS low_sales_low_orders,
FROM segment
--Conclusion: The largest revenue share is generated by customers with above-median average order value and above-median frequency. The second largest share is generated by customers with below-median average order value and above-median frequency.
""").df()

IOException: IO Error: No files found that match the pattern "online_retail.csv"

In [1]:
import matplotlib.pyplot as plt
df.columns = [
    "High Value / High Freq",
    "Low Value / High Freq",
    "High Value / Low Freq",
    "Low Value / Low Freq"]
plt.bar(df.columns, df.iloc[0])
plt.xlabel("Customer Segment")
plt.ylabel("Revenue Share (%)")
plt.title("Revenue Contribution by Customer Segment")
plt.xticks(rotation=45)
for i, v in enumerate(df.iloc[0]):
    plt.text(i, v + 1, f"{v:.1f}%", ha='center')
plt.show()

NameError: name 'df' is not defined